# Allen-Cahn 2D — Learned vs Real Potential (Scatter)

For each snapshot in the test set, compute the scalar potential energy $V(u)$ under two models:

- **Real potential** (`ac_2d` model): $V_\text{real}(u) = \int \frac{1}{2}|\nabla u|^2 + \frac{u^4}{4\varepsilon^2} - \frac{u^2}{2\varepsilon^2}\, dx\, dy$
- **Learned potential** (`s_onsagernet` model): $V_\text{learned}(u) = V_0 + V_1 + V_2$

$V_\text{real}$ is the Lyapunov function for $u_t = \Delta u + (u - u^3)/\varepsilon^2$:
$\delta V/\delta u = -\Delta u + (u^3-u)/\varepsilon^2$, so $\mathrm{d}V/\mathrm{d}t \leq 0$ along trajectories.

In [ ]:
import functools
import glob
import re
from pathlib import Path

import h5py
import hydra
import matplotlib.pyplot as plt
import numpy as np
import rootutils
import seaborn as sns
import torch
import yaml
from omegaconf import OmegaConf
from scipy import stats

ROOT = rootutils.setup_root(".", indicator=".project-root", pythonpath=True)

from src.models.meso_module import MesoLitModule  # noqa: E402

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"ROOT: {ROOT}")
print(f"Device: {device}")

In [ ]:
def find_latest_ckpt(run_dir: Path) -> Path:
    ckpt_dir = run_dir / "checkpoints"
    if ckpt_dir.is_dir():
        ckpts = sorted(
            [p for p in ckpt_dir.glob("*.ckpt") if p.name != "last.ckpt"],
            key=lambda p: p.stat().st_mtime,
        )
        if ckpts:
            return ckpts[-1]
    ckpts = sorted(run_dir.rglob("*.ckpt"), key=lambda p: p.stat().st_mtime)
    if not ckpts:
        raise FileNotFoundError(f"No checkpoint found in {run_dir}")
    return ckpts[-1]


def load_model_for_inference(run_dir: Path, device: str = "cpu") -> MesoLitModule:
    config_file = run_dir / ".hydra" / "config.yaml"
    cfg_text = config_file.read_text()
    # Runs in logs/official/ predate the refactor that moved the spectral
    # OnsagerNet 1d package under dynamics/; rewrite the legacy _target_ path.
    cfg_text = cfg_text.replace(
        "src.models.components.spectral_onsagernet.",
        "src.models.components.dynamics.spectral_onsagernet.",
    )
    cfg_raw = yaml.safe_load(cfg_text)

    dynamics_cfg = OmegaConf.create(cfg_raw["model"]["dynamics"])
    dynamics = hydra.utils.instantiate(dynamics_cfg)

    model_cfg = cfg_raw["model"]
    model = MesoLitModule(
        dynamics=dynamics,
        optimizer=functools.partial(torch.optim.Adam),
        dt=model_cfg["dt"],
        compile=False,
        accumulated_nsteps=model_cfg.get("accumulated_nsteps", 1),
        loss_fn=model_cfg.get("loss_fn", "mse"),
        reg_weight=model_cfg.get("reg_weight"),
    )

    ckpt_path = find_latest_ckpt(run_dir)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    sd = {re.sub(r"\._orig_mod\.", ".", k): v for k, v in ckpt["state_dict"].items()}
    model.load_state_dict(sd, strict=False)
    model.eval().to(device)
    print(f"  Loaded: {ckpt_path.relative_to(ROOT)}")
    return model


def load_test_data(
    data_glob: str,
    split_ratio: tuple = (0.7, 0.2, 0.1),
) -> tuple:
    files = sorted(glob.glob(data_glob))
    trajs, t_coord, x_coord, y_coord = [], None, None, None
    for fp in files:
        with h5py.File(fp, "r") as f:
            u = f["u_sol_all"][:]  # (..., nt, nx, ny, n_vars)
            if t_coord is None and "t_coord" in f:
                t_coord = f["t_coord"][:]
            if x_coord is None and "x_coord" in f:
                x_coord = f["x_coord"][:]
            if y_coord is None and "y_coord" in f:
                y_coord = f["y_coord"][:]
        trajs.append(u.reshape(-1, *u.shape[-4:]))

    u_all = np.concatenate(trajs, axis=0)  # (N, nt, nx, ny, n_vars)
    N = len(u_all)
    val_end = int(N * (split_ratio[0] + split_ratio[1]))
    u_test = u_all[val_end:]  # (N_test, nt, nx, ny, n_vars)
    u_test = np.transpose(u_test, (0, 1, 4, 2, 3))  # → (N_test, nt, n_vars, nx, ny)
    return torch.from_numpy(u_test).float(), t_coord, x_coord, y_coord

In [ ]:
RUNS_BASE = ROOT / "logs/official/runs/ac_2d"

print("Loading 'ac_2d' (real potential) ...")
ac_model = load_model_for_inference(RUNS_BASE / "ac_2d", device=device)

print("Loading 's_onsagernet' (learned potential) ...")
son_model = load_model_for_inference(RUNS_BASE / "s_onsagernet", device=device)

# Expose the potential objects directly
ac_potential = ac_model.dynamics.potential  # AllenCahnPotential (2D)
son_potential = son_model.dynamics.potential  # CoerciveAutogradPotential (2D)
print(f"\nReal potential type   : {type(ac_potential).__name__}")
print(f"Learned potential type: {type(son_potential).__name__}")

In [ ]:
test_data, t_coord, x_coord, y_coord = load_test_data(str(ROOT / "data/allen_cahn_2d/*1000_seed0.hdf5"))
N_test, T, n_vars, Nx, Ny = test_data.shape
print(f"Test set: {N_test} trajectories  |  T={T}, n_vars={n_vars}, Nx={Nx}, Ny={Ny}")

# All snapshots: flatten (N_test, T) → (N_test*T, n_vars, Nx, Ny)
u_all_frames = test_data.reshape(N_test * T, n_vars, Nx, Ny)

In [ ]:
N_frames = N_test * T
BATCH = 128  # batch frames to keep GPU memory bounded for 2D fields

V_real_chunks, V_learned_chunks = [], []
with torch.no_grad():
    for s in range(0, N_frames, BATCH):
        u_b = u_all_frames[s : s + BATCH].to(device)
        bs = u_b.shape[0]

        # AllenCahnPotential (2D).V: expects (B, c, Nx, Ny), returns (B, 1)
        v_real = ac_potential.V(u_b).squeeze(-1)

        # CoerciveAutogradPotential (2D): _V2 returns (B, c, 1) = (B, 1, 1) for c=1,
        # so sum components manually after reshaping to (B, 1).
        _V0 = son_potential._V0(u_b)  # (B, 1)
        _V1 = son_potential._V1(u_b)  # (B, 1)
        _V2 = son_potential._V2(u_b).view(bs, 1)  # (B, 1, 1) → (B, 1)
        v_learned = (_V0 + _V1 + _V2).squeeze(-1)

        V_real_chunks.append(v_real.cpu().numpy())
        V_learned_chunks.append(v_learned.cpu().numpy())

V_real = np.concatenate(V_real_chunks)  # (N_frames,)
V_learned = np.concatenate(V_learned_chunks)  # (N_frames,)

print(f"V_real    shape: {V_real.shape}")
print(f"V_learned shape: {V_learned.shape}")
print(f"V_real   : min={V_real.min():.3f}  max={V_real.max():.3f}  mean={V_real.mean():.3f}")
print(f"V_learned: min={V_learned.min():.3f}  max={V_learned.max():.3f}  mean={V_learned.mean():.3f}")

In [ ]:
# Pearson correlation
r, p_value = stats.pearsonr(V_real, V_learned)
print(f"Pearson r = {r:.4f}  (p = {p_value:.2e})")

# Linear fit: V_learned = slope * V_real + intercept
slope, intercept, *_ = stats.linregress(V_real, V_learned)
print(f"Linear fit: V_learned = {slope:.4f} * V_real + {intercept:.4f}")


N_plot = 10
traj_idx = np.linspace(0, N_test - 1, N_plot, dtype=int)  # evenly spaced subset

colors = sns.color_palette("deep", N_plot)

fig, ax = plt.subplots(figsize=(6, 4))

for plot_i, traj_i in enumerate(traj_idx):
    sl = slice(traj_i * T, traj_i * T + T)
    c = colors[plot_i]
    ax.scatter(V_real[sl], V_learned[sl], s=15, alpha=0.8, marker="+", color=c, edgecolors="none")
    # ax.scatter(
    #     V_real[traj_i * T], V_learned[traj_i * T], s=80, marker="*", color=c, edgecolors="k", linewidths=0.4, zorder=5
    # )

# Linear fit line
vr_range = np.array([V_real.min(), V_real.max()])
ax.plot(
    vr_range,
    slope * vr_range + intercept,
    color="k",
    linewidth=1.2,
    linestyle="--",
    label=r"fit: $V_\theta="
    f"{slope:.2f}"
    r"V+"
    f"{intercept:.2f}"
    r"$",
    alpha=0.5,
)
ax.legend(fontsize=14)

ax.set_xlabel(r"Real potential $V$", fontsize=14)
ax.set_ylabel(r"Learned potential $V_\theta$", fontsize=14)
# ax.set_title(
#     f"Allen-Cahn 2D — Learned vs Real Potential\n"
#     f"(Pearson $r={r:.3f}$, {N_plot} trajectories, ★ = initial states)",
#     fontsize=12,
#     fontweight="bold",
# )
ax.grid(True, alpha=0.3)

plt.tight_layout()
out_path = ROOT / "figs/potential_scatter/ac_2d_potential_scatter.pdf"
plt.savefig(out_path, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path.relative_to(ROOT)}")